# Voice paths test — OmniVoice cloning vs native Amharic speech + Seed-VC

The camille clip's Amharic lines (Camille and Samuel), voiced 12 ways:

| Path | Systems |
|---|---|
| 1. OmniVoice cloning from the speaker's English lines | `omni_today` (today's settings) · `omni_tight` (voice sample with its pauses squeezed out) · `omni_tight_punct` (+ Amharic punctuation `፣ ።`) |
| 2. Native Amharic speech → Seed-VC into the speaker's voice | bases `edge` (edge-tts), `mms` (Meta MMS-TTS), `native` (OmniVoice-Amharic with a native Amharic speaker); each converted by Seed-VC `_xlsr` (fast model) and `_whisper` (quality model) |

Every take is scored: sounds like the speaker, Amharic recognition error, length vs the English line,
pauses per 10 words, and **stray pauses** (pauses where the Amharic has no punctuation).

**Colab**: Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all**. Allow Google Drive when asked.
The inputs are in `MyDrive/LangBridge/experiments/voice_paths` (put there from the PC).

**Kaggle**: add the `voice_paths.zip` dataset, Settings → Accelerator **GPU T4** and **Internet on**, then Run all.

About 30–45 minutes on a T4. Safe to stop: run again and it continues where it stopped.
Results: `results/listen.html` (open it from Drive on the PC to listen side by side), `results/summary.md`.

In [ ]:
#@title Settings
FOLDER = ""  #@param {type:"string"}
#@markdown Empty: `MyDrive/LangBridge/experiments/voice_paths` on Colab, the attached dataset on Kaggle.
SYSTEMS = "all"  #@param {type:"string"}
#@markdown `all`, or names separated by spaces, e.g. `omni_today omni_tight_punct edge edge_whisper`.
SEEDVC_STEPS = 25  #@param {type:"integer"}

In [ ]:
#@title Find the test folder
import glob, os, shutil, subprocess, sys
from pathlib import Path

if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
    if not FOLDER:
        found = glob.glob("/kaggle/input/**/voice_paths/inputs/lines.json", recursive=True)
        if not found:
            raise SystemExit("Attach the voice_paths.zip dataset (Add data) first.")
        src = Path(found[0]).parents[1]
        FOLDER = "/kaggle/working/voice_paths"
        if not Path(FOLDER).exists():
            shutil.copytree(src, FOLDER)  # inputs are read-only on Kaggle; results go to /kaggle/working
    WORK = "/kaggle/working/vp-work"
else:
    from google.colab import drive
    drive.mount("/content/drive")
    FOLDER = FOLDER or "/content/drive/MyDrive/LangBridge/experiments/voice_paths"
    WORK = "/content/vp-work"
if not Path(FOLDER, "inputs", "lines.json").exists():
    raise SystemExit(f"No inputs in {FOLDER}: run spikes/voice_paths/prepare_inputs.py on the PC first.")
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
print("folder:", FOLDER)
print("GPU:", gpu or "none (this will be very slow: pick a T4 runtime)")

In [ ]:
#@title Run the test
cmd = [sys.executable, "-u", f"{FOLDER}/code/voice_paths.py", "--folder", FOLDER, "--work", WORK, "--steps", str(SEEDVC_STEPS)]
if SYSTEMS.strip() and SYSTEMS.strip() != "all":
    cmd += ["--systems", *SYSTEMS.split()]
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    line = line.rstrip().split(chr(13))[-1]
    if line and "%|" not in line:  # progress bars stay out of the log
        print(line, flush=True)
print("exit", p.wait())

In [ ]:
#@title Results
from IPython.display import Markdown, display
res = Path(FOLDER, "results")
if Path(res, "summary.md").exists():
    display(Markdown(Path(res, "summary.md").read_text(encoding="utf-8")))
if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
    shutil.make_archive("/kaggle/working/voice_paths_results", "zip", res)
    print("Download voice_paths_results.zip from the Output panel, unzip it, open listen.html.")
else:
    print(r"On the PC: open G:\My Drive\LangBridge\experiments\voice_paths\results\listen.html")